In [2]:
# --- Step 1: 轻量检查数据的列、行数、示例行（不会把整表载入内存） ---

import pandas as pd
from pathlib import Path

# 路径
csv_path = Path("./data/metadata_merged.csv")
assert csv_path.exists(), f"找不到文件: {csv_path.resolve()}"

# 仅读取表头，获取列名（极快）
header_only = pd.read_csv(csv_path, nrows=0)
cols = list(header_only.columns)

print("\n[INFO] 文件路径:", csv_path.resolve())
print("[INFO] 列名（共 %d 列）:" % len(cols))
print(cols)

# 读取前几行样例，便于肉眼核验字段格式（避免一次性读全表）
sample_n = 5
sample_df = pd.read_csv(csv_path, nrows=sample_n)
print(f"\n[INFO] 前 {sample_n} 行样例：")
print(sample_df.head(sample_n))

# 简单推断：尝试读取少量行做 dtype 预览（避免低内存分块带来的混淆）
preview_n = 1000
preview_df = pd.read_csv(csv_path, nrows=preview_n, low_memory=False)
print(f"\n[INFO] 基于前 {preview_n} 行的 dtype 预览：")
print(preview_df.dtypes)

# 可选：如果数据很大，不建议现在计算完整内存；这里仅估算前1000行的内存占用
print(f"\n[INFO] 前 {preview_n} 行内存占用（近似）：")
print(preview_df.memory_usage(deep=True).sum(), "bytes")



[INFO] 文件路径: /Users/koyo/workspace/recsys/data/metadata_merged.csv
[INFO] 列名（共 31 列）:
['Id', 'Title', 'Slug', 'Tags', 'CreatorUserId', 'OwnerUserId', 'OwnerOrganizationId', 'CurrentDatasetVersionId', 'CurrentDatasourceVersionId', 'ForumId', 'Type', 'CreationDate', 'LastActivityDate', 'TotalViews', 'TotalDownloads', 'TotalVotes', 'TotalKernels', 'Medal', 'MedalAwardDate', 'Subtitle', 'Description', 'CreationDate_dt', 'LastActivityDate_dt', 'age_days', 'days_since_last_activity', 'active_30d', 'has_tags', 'TotalViews_log1p', 'TotalDownloads_log1p', 'TotalVotes_log1p', 'TotalKernels_log1p']

[INFO] 前 5 行样例：
   Id                            Title                            Slug                                            Tags  CreatorUserId  \
0   6   2013 American Community Survey  2013-american-community-survey  computer science, demographics, social science              1   
1   7         May 2015 Reddit Comments        reddit-comments-may-2015       internet, linguistics, online commun

In [6]:
# --- Step 2 (in-memory): 核心字段清洗（不落盘） ---

import re
from pathlib import Path
import pandas as pd
import numpy as np

# 路径
csv_path = Path("./data/metadata_merged.csv")
assert csv_path.exists(), f"找不到文件: {csv_path.resolve()}"

# 仅读取本步需要的列；缺的列用空串占位
need_cols = ["Id","Title","Subtitle","Description","Tags",
             "CreationDate","LastActivityDate","CreationDate_dt","LastActivityDate_dt"]
cols_avail = list(pd.read_csv(csv_path, nrows=0).columns)
usecols = [c for c in need_cols if c in cols_avail]

df = pd.read_csv(csv_path, usecols=usecols, low_memory=False)

# 缺失填空串
for c in ["Title","Subtitle","Description","Tags"]:
    if c not in df.columns:
        df[c] = ""
    else:
        df[c] = df[c].fillna("")

# ---- Tags: 逗号切分 → 小写 → 去重（保序） ----
def _split_tags(s):
    parts = [p.strip().lower() for p in str(s).split(",") if p.strip()]
    seen, out = set(), []
    for p in parts:
        p = re.sub(r"\s+", " ", p)
        if p and p not in seen:
            out.append(p); seen.add(p)
    return out

df["tag_list"] = df["Tags"].map(_split_tags)

# ---- Text: 逐行清洗 Title + Subtitle + Description ----
url_pat = re.compile(r"https?://\S+|www\.\S+")
md_pat  = re.compile(r"\[([^\]]+)\]\([^)]+\)")

def _clean_text_row(row):
    s = " ".join([str(row.get("Title","")), str(row.get("Subtitle","")), str(row.get("Description",""))])
    s = s.replace("\n"," ").replace("\r"," ")
    s = url_pat.sub(" ", s)
    s = md_pat.sub(r"\1", s)
    s = re.sub(r"[^\w\s\-#/@&]+", " ", s)   # 保留少量符号
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

df["text_all"] = df.apply(_clean_text_row, axis=1)

# ---- 时间字段：优先 *_dt，回退到原始字符串列 ----
def _coalesce_datetime(primary, fallback):
    if primary in df.columns:
        p = pd.to_datetime(df[primary], errors="coerce")
        if fallback in df.columns:
            return p.fillna(pd.to_datetime(df[fallback], errors="coerce"))
        return p
    else:
        return pd.to_datetime(df.get(fallback, pd.Series([None]*len(df))), errors="coerce")

df["created_at"]     = _coalesce_datetime("CreationDate_dt", "CreationDate")
df["last_active_at"] = _coalesce_datetime("LastActivityDate_dt", "LastActivityDate")

# ---- 去重与编号 ----
df = df.drop_duplicates(subset=["Id"]).reset_index(drop=True)
df["doc_idx"] = np.arange(len(df), dtype=np.int64)

# 形成核心文档表（用于后续所有视图）
doc_df = df[["doc_idx","Id","text_all","tag_list","created_at","last_active_at"]].copy()

# （可选）便于后续映射：内存里保留一个索引映射 DataFrame / 字典
index_map = doc_df[["doc_idx","Id"]].copy()
id2idx = dict(zip(index_map["Id"].tolist(), index_map["doc_idx"].tolist()))

# ---- 基本统计与样例 ----
n_docs   = len(doc_df)
has_tags = (doc_df["tag_list"].str.len() > 0).sum()
avg_tags = doc_df.loc[doc_df["tag_list"].str.len() > 0, "tag_list"].map(len).mean() if has_tags>0 else 0.0
avg_text = int(doc_df["text_all"].map(lambda s: len(str(s).split())).mean())

print("[INFO] 文档数:", n_docs)
print("[INFO] 至少1个标签的文档数/占比: {}/{} ({:.1f}%)".format(has_tags, n_docs, 100*has_tags/max(n_docs,1)))
print("[INFO] 平均标签数(有标签样本): {:.2f}".format(avg_tags))
print("[INFO] 平均文本词数(粗略):", avg_text)

print("\n[INFO] 样例行（前3）：")
print(doc_df.head(3).to_string(index=False))

# 关键变量在内存中：
# - doc_df: 后续视图构建的唯一真源（包含 text_all / tag_list / 时间 / doc_idx）
# - index_map / id2idx: 便于回溯 Id 与矩阵索引的对应


[INFO] 文档数: 521735
[INFO] 至少1个标签的文档数/占比: 214603/521735 (41.1%)
[INFO] 平均标签数(有标签样本): 2.08
[INFO] 平均文本词数(粗略): 38

[INFO] 样例行（前3）：
 doc_idx  Id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [7]:
# --- Step 3: Tag 视图（D–T）构筑：vocab/CSR + TF-IDF + PPMI（全在内存） ---

import numpy as np
from collections import Counter
from scipy import sparse

# 先决条件：上一单元生成的 doc_df 在内存中，包含列：
# - 'doc_idx'（0..N-1）、'tag_list'（list[str]）
assert 'doc_df' in globals(), "未发现 doc_df，请先运行清洗步骤（Step 2 in-memory）。"

# ========== 可调参数 ==========
MIN_DF   = 10      # 过滤低文档频率的标签（可调：5/10/20）
MAX_VOCAB = None   # 可选限制词表上限（如 100_000），默认不过滤
ROW_NORM = True    # 是否对加权矩阵做逐行 L2 归一
# ============================

N = len(doc_df)

# 1) 统计标签的文档频率（DF）
tag_df_counter = Counter()
for tags in doc_df['tag_list']:
    if isinstance(tags, list) and tags:
        tag_df_counter.update(set(tags))   # DF（按文档去重）

raw_vocab_size = len(tag_df_counter)
kept = [(t, c) for t, c in tag_df_counter.items() if c >= MIN_DF]
kept.sort(key=lambda x: (-x[1], x[0]))
if MAX_VOCAB is not None:
    kept = kept[:MAX_VOCAB]

tag2id = {t:i for i,(t,_) in enumerate(kept)}
id2tag = [t for t,_ in kept]
df_counts = np.array([c for _,c in kept], dtype=np.int64)
V = len(tag2id)

print(f"[TAG] docs={N:,}, raw_unique_tags={raw_vocab_size:,} -> kept(min_df>={MIN_DF})={V:,}")

# 2) 构建 D–T（binary）CSR
rows, cols, data = [], [], []
for r, tags in enumerate(doc_df['tag_list']):
    if not tags: 
        continue
    for t in set(tags):
        j = tag2id.get(t)
        if j is not None:
            rows.append(r); cols.append(j); data.append(1)

if not data:
    raise RuntimeError("没有标签通过 min_df 过滤；请降低 MIN_DF。")

rows = np.asarray(rows, dtype=np.int64)
cols = np.asarray(cols, dtype=np.int64)
data = np.asarray(data, dtype=np.float32)

DT_bin = sparse.csr_matrix((data, (rows, cols)), shape=(N, V))
nnz = DT_bin.nnz
density = nnz / (N * V) if V > 0 else 0.0
docs_with_kept_tags = int((DT_bin.getnnz(axis=1) > 0).sum())

print(f"[TAG] D–T nnz={nnz:,}, density={density:.6e}, docs_with_kept_tags={docs_with_kept_tags:,} ({docs_with_kept_tags/N:.1%})")
print("[TAG] top-15 tags by df:")
topk = min(15, V)
top_idx = np.argsort(-df_counts)[:topk]
for idx in top_idx:
    print(f"   {id2tag[idx]:<30s} df={df_counts[idx]}")

# 3) TF-IDF 加权（平滑 idf）
df_vec = np.asarray(DT_bin.astype(bool).sum(axis=0)).ravel().astype(np.int64)
idf = np.log((1 + N) / (1 + df_vec)) + 1.0   # smooth idf
tfidf_data = idf[cols].astype(np.float32)    # tf=1（标签去重后为 binary）
DT_tfidf = sparse.csr_matrix((tfidf_data, (rows, cols)), shape=(N, V))

if ROW_NORM:
    row_norms = np.sqrt(np.asarray(DT_tfidf.power(2).sum(axis=1)).ravel()) + 1e-12
    DT_tfidf = DT_tfidf.multiply(1.0 / row_norms[:, None])

print(f"[TAG] TF-IDF done. row_norm check (min/median/max): "
      f"{float(np.min(np.asarray(DT_tfidf.power(2).sum(axis=1)))):.4f}/"
      f"{float(np.median(np.asarray(DT_tfidf.power(2).sum(axis=1)))):.4f}/"
      f"{float(np.max(np.asarray(DT_tfidf.power(2).sum(axis=1)))):.4f}")

# 4) PPMI 加权（正点互信息）
# PMI(d,t)=log((x_dt * Npairs)/(row_sum_d * col_sum_t)), x_dt=1；PPMI=max(PMI,0)
row_sum = np.asarray(DT_bin.sum(axis=1)).ravel().astype(np.float64)  # 每文档标签数
col_sum = np.asarray(DT_bin.sum(axis=0)).ravel().astype(np.float64)  # 每标签 DF
Npairs  = float(nnz)

num = np.ones_like(data, dtype=np.float64) * Npairs
den = row_sum[rows] * col_sum[cols]
ppmi_vals = np.maximum(0.0, np.log(num / (den + 1e-12))).astype(np.float32)
DT_ppmi = sparse.csr_matrix((ppmi_vals, (rows, cols)), shape=(N, V))

if ROW_NORM:
    ppmi_row_norms = np.sqrt(np.asarray(DT_ppmi.power(2).sum(axis=1)).ravel()) + 1e-12
    DT_ppmi = DT_ppmi.multiply(1.0 / ppmi_row_norms[:, None])

print(f"[TAG] PPMI done. nnz={DT_ppmi.nnz:,}, ppmi(mean/std)={ppmi_vals.mean():.4f}/{ppmi_vals.std():.4f}")

# 5) 关键对象留在内存（供后续用）
# - tag2id / id2tag / df_counts
# - DT_bin（binary），DT_tfidf（行归一），DT_ppmi（行归一）


[TAG] docs=521,735, raw_unique_tags=597 -> kept(min_df>=10)=394
[TAG] D–T nnz=445,426, density=2.166853e-03, docs_with_kept_tags=214,585 (41.1%)
[TAG] top-15 tags by df:
   pre-trained model              df=30498
   business                       df=27014
   earth and nature               df=17261
   computer science               df=12007
   arts and entertainment         df=11738
   tabular                        df=11589
   education                      df=8809
   data analytics                 df=8283
   image                          df=8064
   beginner                       df=8030
   health                         df=7581
   text                           df=7424
   data visualization             df=7388
   movies and tv shows            df=6886
   finance                        df=5731
[TAG] TF-IDF done. row_norm check (min/median/max): 0.0000/0.0000/1.0000
[TAG] PPMI done. nnz=445,426, ppmi(mean/std)=3.7901/1.3509


In [8]:
# --- Step 4: Text 视图（D–W）构筑：BM25 加权 CSR（内存态） ---

import re
import numpy as np
from collections import Counter
from scipy import sparse

# 先决条件：doc_df 在内存，含 ['doc_idx','text_all']
assert 'doc_df' in globals(), "未发现 doc_df，请先运行清洗步骤。"

# ========== 可调参数 ==========
MIN_DF_W       = 200        # 词项最小文档频率阈值（可按机器内存调大：200/300/500）
MAX_DF_RATIO_W = 0.50       # 过滤过于常见的词（>50% 文档出现的词去掉）
MAX_VOCAB_W    = 100_000    # 词表上限（None 表示不限；建议 50k~150k）
TOKEN_MIN_LEN  = 2          # 最小 token 长度
K1, B          = 1.2, 0.75  # BM25 参数
ROW_NORM       = True       # 是否对行做 L2 归一（便于后续相似度）
PROGRESS_EVERY = 50_000     # 进度打印步长
# ============================

N = len(doc_df)
token_pat = re.compile(r"[a-z0-9]+")

def tokenize(s):
    # text_all 已经 lower；只取字母数字序列，过滤过短 token
    toks = token_pat.findall(str(s))
    return [t for t in toks if len(t) >= TOKEN_MIN_LEN]

# ---------- Pass 1: 统计文档频率（DF） ----------
df_counter = Counter()
for i, txt in enumerate(doc_df['text_all'].values):
    if i % PROGRESS_EVERY == 0 and i > 0:
        print(f"[TEXT:DF] processed {i:,}/{N:,}")
    toks = tokenize(txt)
    if not toks:
        continue
    uniq = set(toks)
    df_counter.update(uniq)

raw_vocab = len(df_counter)
max_df_abs = int(MAX_DF_RATIO_W * N)

kept = [(t, c) for t, c in df_counter.items() if c >= MIN_DF_W and c <= max_df_abs]
kept.sort(key=lambda x: (-x[1], x[0]))
if MAX_VOCAB_W is not None:
    kept = kept[:MAX_VOCAB_W]

word2id = {w:i for i,(w,_) in enumerate(kept)}
id2word = [w for w,_ in kept]
df_counts_w = np.array([c for _,c in kept], dtype=np.int64)
V = len(word2id)

print(f"[TEXT] docs={N:,}, raw_vocab={raw_vocab:,} -> kept(min_df>={MIN_DF_W}, max_df<={MAX_DF_RATIO_W:.0%})={V:,}")

# ---------- BM25: 需要先得到每篇文档的长度（基于“保留的词”） ----------
doc_len = np.zeros(N, dtype=np.int32)
for i, txt in enumerate(doc_df['text_all'].values):
    if i % PROGRESS_EVERY == 0 and i > 0:
        print(f"[TEXT:LEN] processed {i:,}/{N:,}")
    toks = tokenize(txt)
    if not toks:
        continue
    # 仅保留词表里的词
    cnt = 0
    for t in toks:
        if t in word2id:
            cnt += 1
    doc_len[i] = cnt

avg_len = float(doc_len.mean() if N > 0 else 0.0)
print(f"[TEXT] avg_doc_len (kept-vocab based) = {avg_len:.2f}")

# ---------- 计算 idf（BM25 版本，clamp>=0） ----------
# idf = log( (N - df + 0.5) / (df + 0.5) )
idf = np.log((N - df_counts_w + 0.5) / (df_counts_w + 0.5))
idf = np.maximum(idf, 0.0).astype(np.float32)

# ---------- Pass 2: 构建 D–W 的 BM25 加权 CSR ----------
rows, cols, data = [], [], []
for i, txt in enumerate(doc_df['text_all'].values):
    if i % PROGRESS_EVERY == 0 and i > 0:
        print(f"[TEXT:CSR] processed {i:,}/{N:,}")
    toks = tokenize(txt)
    if not toks:
        continue
    # 统计该文档的 TF（仅对保留词）
    tf_local = {}
    for t in toks:
        j = word2id.get(t)
        if j is not None:
            tf_local[j] = tf_local.get(j, 0) + 1

    if not tf_local:
        continue

    Ld = max(doc_len[i], 1)
    norm = (1 - B) + B * (Ld / max(avg_len, 1e-6))

    for j, tf in tf_local.items():
        w = idf[j] * (tf * (K1 + 1)) / (tf + K1 * norm)
        if w <= 0:
            continue
        rows.append(i); cols.append(j); data.append(w)

if not data:
    raise RuntimeError("D–W 矩阵为空；请降低 MIN_DF_W 或提高 MAX_DF_RATIO_W。")

rows = np.asarray(rows, dtype=np.int64)
cols = np.asarray(cols, dtype=np.int64)
data = np.asarray(data, dtype=np.float32)

DW_bm25 = sparse.csr_matrix((data, (rows, cols)), shape=(N, V))

if ROW_NORM:
    row_norms = np.sqrt(np.asarray(DW_bm25.power(2).sum(axis=1)).ravel()) + 1e-12
    DW_bm25 = DW_bm25.multiply(1.0 / row_norms[:, None])

nnz = DW_bm25.nnz
density = nnz / (N * V) if V > 0 else 0.0
docs_with_words = int((DW_bm25.getnnz(axis=1) > 0).sum())

print(f"[TEXT] D–W nnz={nnz:,}, density={density:.6e}, docs_with_words={docs_with_words:,} ({docs_with_words/N:.1%})")
print("[TEXT] top-15 words by df:")
topk = min(15, V)
top_idx = np.argsort(-df_counts_w)[:topk]
for idx in top_idx:
    print(f"   {id2word[idx]:<20s} df={df_counts_w[idx]}")

print(f"[TEXT] BM25 done. row_norm check (min/median/max): "
      f"{float(np.min(np.asarray(DW_bm25.power(2).sum(axis=1)))):.4f}/"
      f"{float(np.median(np.asarray(DW_bm25.power(2).sum(axis=1)))):.4f}/"
      f"{float(np.max(np.asarray(DW_bm25.power(2).sum(axis=1)))):.4f}")


[TEXT:DF] processed 50,000/521,735
[TEXT:DF] processed 100,000/521,735
[TEXT:DF] processed 150,000/521,735
[TEXT:DF] processed 200,000/521,735
[TEXT:DF] processed 250,000/521,735
[TEXT:DF] processed 300,000/521,735
[TEXT:DF] processed 350,000/521,735
[TEXT:DF] processed 400,000/521,735
[TEXT:DF] processed 450,000/521,735
[TEXT:DF] processed 500,000/521,735
[TEXT] docs=521,735, raw_vocab=372,377 -> kept(min_df>=200, max_df<=50%)=5,714
[TEXT:LEN] processed 50,000/521,735
[TEXT:LEN] processed 100,000/521,735
[TEXT:LEN] processed 150,000/521,735
[TEXT:LEN] processed 200,000/521,735
[TEXT:LEN] processed 250,000/521,735
[TEXT:LEN] processed 300,000/521,735
[TEXT:LEN] processed 350,000/521,735
[TEXT:LEN] processed 400,000/521,735
[TEXT:LEN] processed 450,000/521,735
[TEXT:LEN] processed 500,000/521,735
[TEXT] avg_doc_len (kept-vocab based) = 32.77
[TEXT:CSR] processed 50,000/521,735
[TEXT:CSR] processed 100,000/521,735
[TEXT:CSR] processed 150,000/521,735
[TEXT:CSR] processed 200,000/521,735


In [14]:
# --- Step 5-Fix2: 稳健加权采样 + 重新生成随机游走语料（D-only） ---

import numpy as np
from scipy import sparse

# 前置检查与格式统一
for name in ["doc_df", "DT_ppmi", "DW_bm25"]:
    assert name in globals(), f"未发现 {name}，请先运行前置步骤。"
DT_ppmi = DT_ppmi.tocsr().astype(np.float32, copy=False)
DW_bm25 = DW_bm25.tocsr().astype(np.float32, copy=False)

# 参数（可保持与上次一致）
R_WALKS_PER_DOC = 2
L_DOCS_PER_SENT = 20
SEED = 42
VIEW_SELECT = ("tag", "text")
AVOID_BACKTRACK = True
MIN_ROW_DEG = 1
PROGRESS_EVERY = 50_000

rng = np.random.default_rng(SEED)

def _row_neighbors_csr(mat_csr: sparse.csr_matrix, r: int):
    start, end = mat_csr.indptr[r], mat_csr.indptr[r+1]
    if start == end:
        return None, None
    return mat_csr.indices[start:end], mat_csr.data[start:end]

def _sample_from(weights):
    """
    稳健的加权采样：
    - 使用未归一化权重（float64）构造累积和
    - 以 total*U(0,1) 作为 target，side='left' 查找
    - 超界时钳制到最后一个位置
    """
    if weights is None or len(weights) == 0:
        return None
    w = np.asarray(weights, dtype=np.float64)
    total = w.sum()
    if not np.isfinite(total) or total <= 0:
        return None
    cdf = np.cumsum(w)
    target = rng.random() * total  # [0, total)
    pos = int(np.searchsorted(cdf, target, side="left"))
    if pos >= cdf.size:  # 极端浮点情况，钳制
        pos = cdf.size - 1
    return pos

def generate_sentences_for_view(DX: sparse.csr_matrix,
                                name: str,
                                r_per_doc: int,
                                L_docs: int,
                                avoid_backtrack: bool = True):
    N, V = DX.shape
    deg_D = np.asarray(DX.getnnz(axis=1)).ravel()
    XD = DX.transpose().tocsr()

    start_docs = np.where(deg_D >= MIN_ROW_DEG)[0]
    n_starts = len(start_docs)
    if n_starts == 0:
        raise RuntimeError(f"[{name}] 没有可用起点（行度为0）。")

    sentences = []
    print(f"[RW-{name}] starts={n_starts:,}, target_walks≈{n_starts*r_per_doc:,}, L_docs={L_docs}")

    for s_idx, d0 in enumerate(start_docs):
        if (s_idx + 1) % PROGRESS_EVERY == 0:
            print(f"[RW-{name}] progressed: {s_idx+1:,}/{n_starts:,}, sentences={len(sentences):,}")

        for _ in range(r_per_doc):
            seq = [int(d0)]
            prev_d = None
            cur_d = int(d0)

            for _ in range(L_docs - 1):
                # D -> X
                x_cols, x_w = _row_neighbors_csr(DX, cur_d)
                if x_cols is None:
                    break
                pos = _sample_from(x_w)
                if pos is None:
                    break
                x = int(x_cols[pos])

                # X -> D
                d_rows, d_w = _row_neighbors_csr(XD, x)
                if d_rows is None:
                    break

                if avoid_backtrack and prev_d is not None and d_rows.size > 1:
                    d_w_local = d_w.copy()
                    mask = (d_rows == prev_d)
                    if mask.any():
                        d_w_local[mask] = 0.0
                    pos2 = _sample_from(d_w_local)
                else:
                    pos2 = _sample_from(d_w)

                if pos2 is None:
                    break

                # 保险起见再钳制一次
                if pos2 >= d_rows.size:
                    pos2 = d_rows.size - 1

                next_d = int(d_rows[pos2])
                seq.append(next_d)
                prev_d, cur_d = cur_d, next_d

            if len(seq) >= 2:
                sentences.append(seq)

    avg_len = float(np.mean([len(s) for s in sentences])) if sentences else 0.0
    print(f"[RW-{name}] done. sentences={len(sentences):,}, avg_len={avg_len:.2f}")
    return sentences

def coverage_stats(sents, name):
    if not sents:
        print(f"[RW-{name}] empty.")
        return
    seen = set()
    total_tokens = 0
    for s in sents:
        seen.update(s)
        total_tokens += len(s)
    print(f"[RW-{name}] unique_docs_in_corpus={len(seen):,} "
          f"({len(seen)/len(doc_df):.1%}), total_D_tokens={total_tokens:,}")

# 重新生成
sentences_tag = generate_sentences_for_view(DT_ppmi, "tag",
                                            r_per_doc=R_WALKS_PER_DOC,
                                            L_docs=L_DOCS_PER_SENT,
                                            avoid_backtrack=AVOID_BACKTRACK)
sentences_text = generate_sentences_for_view(DW_bm25, "text",
                                             r_per_doc=R_WALKS_PER_DOC,
                                             L_docs=L_DOCS_PER_SENT,
                                             avoid_backtrack=AVOID_BACKTRACK)

coverage_stats(sentences_tag,  "tag")
coverage_stats(sentences_text, "text")


[RW-tag] starts=214,585, target_walks≈429,170, L_docs=20
[RW-tag] progressed: 50,000/214,585, sentences=99,998
[RW-tag] progressed: 100,000/214,585, sentences=199,998
[RW-tag] progressed: 150,000/214,585, sentences=299,998
[RW-tag] progressed: 200,000/214,585, sentences=399,998
[RW-tag] done. sentences=429,170, avg_len=20.00
[RW-text] starts=417,204, target_walks≈834,408, L_docs=20
[RW-text] progressed: 50,000/417,204, sentences=99,998
[RW-text] progressed: 100,000/417,204, sentences=199,998
[RW-text] progressed: 150,000/417,204, sentences=299,998
[RW-text] progressed: 200,000/417,204, sentences=399,998
[RW-text] progressed: 250,000/417,204, sentences=499,998
[RW-text] progressed: 300,000/417,204, sentences=599,998
[RW-text] progressed: 350,000/417,204, sentences=699,998
[RW-text] progressed: 400,000/417,204, sentences=799,998
[RW-text] done. sentences=834,408, avg_len=20.00
[RW-tag] unique_docs_in_corpus=214,585 (41.1%), total_D_tokens=8,583,400
[RW-text] unique_docs_in_corpus=417,204

In [17]:
# --- Step 5-Inspect: 抽样打印 D–X 与 D-only 序列（可读化质检） ---

import numpy as np
from scipy import sparse

# 先决条件
for name in ["doc_df", "DT_ppmi", "DW_bm25", "id2tag", "id2word"]:
    assert name in globals(), f"未发现 {name}，请先运行前置步骤。"

DT_ppmi = DT_ppmi.tocsr().astype(np.float32, copy=False)
DW_bm25 = DW_bm25.tocsr().astype(np.float32, copy=False)

SEED = 7
rng = np.random.default_rng(SEED)

def _row_neighbors_csr(mat: sparse.csr_matrix, r: int):
    a, b = mat.indptr[r], mat.indptr[r+1]
    if a == b:
        return None, None
    return mat.indices[a:b], mat.data[a:b]

def _sample_from(weights):
    if weights is None or len(weights) == 0:
        return None
    w = np.asarray(weights, dtype=np.float64)
    tot = w.sum()
    if not np.isfinite(tot) or tot <= 0:
        return None
    cdf = np.cumsum(w)
    target = rng.random() * tot
    pos = int(np.searchsorted(cdf, target, side="left"))
    if pos >= cdf.size:
        pos = cdf.size - 1
    return pos

def sample_walks_with_DX(DX: sparse.csr_matrix, id2x, view_name, num_walks=3, L_docs=8, avoid_backtrack=True):
    N, V = DX.shape
    XD = DX.transpose().tocsr()
    deg_D = np.asarray(DX.getnnz(axis=1)).ravel()
    starts = np.where(deg_D > 0)[0]
    if starts.size == 0:
        print(f"[INSPECT-{view_name}] 无可用起点。")
        return

    pick = rng.choice(starts, size=min(num_walks, starts.size), replace=False)
    id_arr = doc_df["Id"].to_numpy()

    print(f"\n[INSPECT-{view_name}] 随机抽样 {len(pick)} 条（每条 D 长度≈{L_docs}）\n")

    for idx, d0 in enumerate(pick, 1):
        d_seq = [int(d0)]
        dx_pairs = []  # [(D, X), (D, X), ...] 最后一个 D 另行追加
        prev_d = None
        cur_d = int(d0)

        for _ in range(L_docs - 1):
            # D -> X
            x_cols, x_w = _row_neighbors_csr(DX, cur_d)
            if x_cols is None:
                break
            pos = _sample_from(x_w)
            if pos is None:
                break
            x = int(x_cols[pos])

            # X -> D
            d_rows, d_w = _row_neighbors_csr(XD, x)
            if d_rows is None:
                break

            if avoid_backtrack and prev_d is not None and d_rows.size > 1:
                d_w_local = d_w.copy()
                mask = (d_rows == prev_d)
                if mask.any():
                    d_w_local[mask] = 0.0
                pos2 = _sample_from(d_w_local)
            else:
                pos2 = _sample_from(d_w)
            if pos2 is None:
                break

            next_d = int(d_rows[pos2])
            dx_pairs.append((cur_d, x))
            d_seq.append(next_d)
            prev_d, cur_d = cur_d, next_d

        # 打印 D–X–D（可读化）与 D-only 两种
        def d_str(d):
            return f"D(doc_idx={d}, Id={id_arr[d]})"
        if view_name == "tag":
            def x_str(x): return f"T('{id2x[x]}')"
        else:
            def x_str(x): return f"W('{id2x[x]}')"

        # 拼接成 D0 -> X0 -> D1 -> X1 -> ... -> Dm 的字符串
        parts = []
        for (d, x) in dx_pairs:
            parts.append(d_str(d)); parts.append(x_str(x))
        parts.append(d_str(d_seq[-1]))  # 末尾的 Dm

        print(f"  示例 #{idx}")
        print("  D–X–D:", "  ->  ".join(parts[: 2*L_docs-1]))  # 截断防过长
        # D-only（前 20 个）
        print("  D-only:", d_seq[:20], "\n")

# 抽样打印 Tag 与 Text 视图各 3 条
sample_walks_with_DX(DT_ppmi, id2tag, view_name="tag",  num_walks=3, L_docs=8, avoid_backtrack=True)
sample_walks_with_DX(DW_bm25, id2word, view_name="text", num_walks=3, L_docs=8, avoid_backtrack=True)



[INSPECT-tag] 随机抽样 3 条（每条 D 长度≈8）

  示例 #1
  D–X–D: D(doc_idx=489134, Id=7598502)  ->  T('retail and shopping')  ->  D(doc_idx=128439, Id=1920551)  ->  T('art')  ->  D(doc_idx=1088, Id=1365)  ->  T('art')  ->  D(doc_idx=397043, Id=6094339)  ->  T('art')  ->  D(doc_idx=128392, Id=1920073)  ->  T('retail and shopping')  ->  D(doc_idx=106472, Id=1611876)  ->  T('retail and shopping')  ->  D(doc_idx=220526, Id=3345202)  ->  T('retail and shopping')  ->  D(doc_idx=518482, Id=8040521)
  D-only: [489134, 128439, 1088, 397043, 128392, 106472, 220526, 518482] 

  示例 #2
  D–X–D: D(doc_idx=305677, Id=4691686)  ->  T('arts and entertainment')  ->  D(doc_idx=292074, Id=4458934)  ->  T('arts and entertainment')  ->  D(doc_idx=75085, Id=1232923)  ->  T('arts and entertainment')  ->  D(doc_idx=286584, Id=4378634)  ->  T('arts and entertainment')  ->  D(doc_idx=13422, Id=107340)  ->  T('text')  ->  D(doc_idx=196135, Id=2941611)  ->  T('ratings and reviews')  ->  D(doc_idx=395842, Id=6073043)  ->  T('r

In [20]:
# --- Step 6a: WS-SGNS 训练（Tag 视图，基于 D-only 语料） ---

import os, math, numpy as np

# 先决条件
for name in ["doc_df", "sentences_tag"]:
    assert name in globals(), f"未发现 {name}，请先运行前置步骤。"

# 参数（先轻量跑一版，确认 OK 后可加大 epochs 等）
SGNS_DIM      = 128     # 向量维度，可后续调到 256
SGNS_WINDOW   = 5       # 上下文窗口
SGNS_NEGATIVE = 10      # 负采样个数
SGNS_EPOCHS   = 2       # 先跑 2 轮快速验证；确认后可增至 5~10
SGNS_SEED     = 42
SGNS_SAMPLE   = 1e-3    # 高频下采样
WORKERS       = max(1, os.cpu_count() or 1)

# 将 doc_idx 序列转为字符串 token（gensim 词表键必须是 str）
class DocIdxSentences:
    def __init__(self, sents):
        self.sents = sents
    def __iter__(self):
        for s in self.sents:
            # 注意：s 是 List[int]，这里转为字符串，避免 int 键
            yield [str(int(x)) for x in s]
    def __len__(self):
        return len(self.sents)

tag_corpus = DocIdxSentences(sentences_tag)

# 尝试导入 gensim；若无则提示安装后重试
try:
    from gensim.models import Word2Vec
except Exception as e:
    print("[ERROR] 未检测到 gensim。请先安装后重试：")
    print("  pip install -U gensim==4.3.2")
    raise

print(f"[SGNS-tag] training on {len(tag_corpus):,} sentences "
      f"(avg len≈{np.mean([len(s) for s in sentences_tag]):.1f}), "
      f"dim={SGNS_DIM}, window={SGNS_WINDOW}, negative={SGNS_NEGATIVE}, epochs={SGNS_EPOCHS}, workers={WORKERS}")

# 训练 SGNS（gensim 会自动 build_vocab + train）
model_tag = Word2Vec(
    sentences=tag_corpus,
    vector_size=SGNS_DIM,
    window=SGNS_WINDOW,
    sg=1,                # skip-gram
    negative=SGNS_NEGATIVE,
    hs=0,
    min_count=1,         # 每个 doc_idx 都需要向量
    workers=WORKERS,
    epochs=SGNS_EPOCHS,
    sample=SGNS_SAMPLE,
    seed=SGNS_SEED,
)

# 提取向量并做 L2 归一
N = len(doc_df)
Z_tag = np.zeros((N, SGNS_DIM), dtype=np.float32)

# gensim 的词表
keys = model_tag.wv.key_to_index
hit = 0
for d in range(N):
    tok = str(d)
    if tok in keys:
        Z_tag[d] = model_tag.wv[tok]
        hit += 1

# 行归一
norms = np.linalg.norm(Z_tag, axis=1, keepdims=True)
mask = norms.squeeze() > 0
Z_tag[mask] = Z_tag[mask] / norms[mask]

print(f"[SGNS-tag] vectors built: covered_docs={hit}/{N} ({hit/max(N,1):.1%}), "
      f"nonzero_rows={int(mask.sum())}")

# 基础健诊：随机抽 3 个已覆盖 doc，打印最相似的 5 个（仅 sanity check）
try:
    import random
    samp = random.sample([i for i in range(N) if str(i) in keys], k=min(3, hit))
    for idx in samp:
        tok = str(idx)
        sims = model_tag.wv.most_similar(tok, topn=5)
        print(f"[SGNS-tag] sample NN for doc_idx={idx}: {[(int(t), round(float(s),4)) for t,s in sims]}")
except Exception as e:
    print("[SGNS-tag] NN sanity check skipped:", e)

# 关键产物留在内存：
# - Z_tag: (N, d) 的数据集嵌入（L2 归一），后续将用于构造 S^(tag)


[SGNS-tag] training on 429,170 sentences (avg len≈20.0), dim=128, window=5, negative=10, epochs=2, workers=10
[SGNS-tag] vectors built: covered_docs=214585/521735 (41.1%), nonzero_rows=214585
[SGNS-tag] sample NN for doc_idx=344084: [(57822, 0.9992), (278337, 0.9992), (162323, 0.9992), (394828, 0.9992), (107917, 0.9992)]
[SGNS-tag] sample NN for doc_idx=172772: [(96854, 0.9993), (424108, 0.9993), (155412, 0.9993), (494955, 0.9993), (307852, 0.9993)]
[SGNS-tag] sample NN for doc_idx=418998: [(193918, 0.9993), (394237, 0.9993), (512448, 0.9993), (129360, 0.9993), (237308, 0.9993)]


In [21]:
# --- Step 6b: WS-SGNS 训练（Text 视图，基于 D-only 语料） ---

import os, math, numpy as np

# 前置
for name in ["doc_df", "sentences_text"]:
    assert name in globals(), f"未发现 {name}，请先运行前置步骤。"

# 与 Tag 视图保持一致的轻量配置（验证流程）。确认 OK 后可把 epochs 提到 5~10。
SGNS_DIM      = 128
SGNS_WINDOW   = 5
SGNS_NEGATIVE = 10
SGNS_EPOCHS   = 2        # 先快速跑通
SGNS_SEED     = 43       # 与 tag 视图错开一个种子
SGNS_SAMPLE   = 1e-3
WORKERS       = max(1, os.cpu_count() or 1)

# 将 doc_idx 序列转为字符串 token
class DocIdxSentencesText:
    def __init__(self, sents):
        self.sents = sents
    def __iter__(self):
        for s in self.sents:
            yield [str(int(x)) for x in s]
    def __len__(self):
        return len(self.sents)

text_corpus = DocIdxSentencesText(sentences_text)

# 导入 gensim（前一步已安装则可复用）
try:
    from gensim.models import Word2Vec
except Exception as e:
    print("[ERROR] 未检测到 gensim。请先安装后重试： pip install -U gensim==4.3.2")
    raise

print(f"[SGNS-text] training on {len(text_corpus):,} sentences "
      f"(avg len≈{np.mean([len(s) for s in sentences_text]):.1f}), "
      f"dim={SGNS_DIM}, window={SGNS_WINDOW}, negative={SGNS_NEGATIVE}, epochs={SGNS_EPOCHS}, workers={WORKERS}")

model_text = Word2Vec(
    sentences=text_corpus,
    vector_size=SGNS_DIM,
    window=SGNS_WINDOW,
    sg=1,
    negative=SGNS_NEGATIVE,
    hs=0,
    min_count=1,
    workers=WORKERS,
    epochs=SGNS_EPOCHS,
    sample=SGNS_SAMPLE,
    seed=SGNS_SEED,
)

# 提取向量并做 L2 归一
N = len(doc_df)
Z_text = np.zeros((N, SGNS_DIM), dtype=np.float32)
keys_t = model_text.wv.key_to_index
hit_t = 0
for d in range(N):
    tok = str(d)
    if tok in keys_t:
        Z_text[d] = model_text.wv[tok]
        hit_t += 1

# 行归一
norms_t = np.linalg.norm(Z_text, axis=1, keepdims=True)
mask_t = norms_t.squeeze() > 0
Z_text[mask_t] = Z_text[mask_t] / norms_t[mask_t]

print(f"[SGNS-text] vectors built: covered_docs={hit_t}/{N} ({hit_t/max(N,1):.1%}), "
      f"nonzero_rows={int(mask_t.sum())}")

# 随机抽 3 个已覆盖 doc，打印最相似的 5 个（sanity check）
try:
    import random
    samp = random.sample([i for i in range(N) if str(i) in keys_t], k=min(3, hit_t))
    for idx in samp:
        tok = str(idx)
        sims = model_text.wv.most_similar(tok, topn=5)
        print(f"[SGNS-text] sample NN for doc_idx={idx}: {[(int(t), round(float(s),4)) for t,s in sims]}")
except Exception as e:
    print("[SGNS-text] NN sanity check skipped:", e)

# 关键产物在内存：
# - Z_text: (N, d) 的数据集向量（L2 归一）


[SGNS-text] training on 834,408 sentences (avg len≈20.0), dim=128, window=5, negative=10, epochs=2, workers=10
[SGNS-text] vectors built: covered_docs=417204/521735 (80.0%), nonzero_rows=417204
[SGNS-text] sample NN for doc_idx=248472: [(436290, 0.9993), (238912, 0.9993), (458084, 0.9993), (271833, 0.9993), (398130, 0.9993)]
[SGNS-text] sample NN for doc_idx=185328: [(199631, 0.9994), (100811, 0.9993), (260570, 0.9993), (125375, 0.9993), (148985, 0.9993)]
[SGNS-text] sample NN for doc_idx=57264: [(92259, 0.9993), (499391, 0.9993), (501809, 0.9993), (297496, 0.9993), (9748, 0.9993)]
